MemCtrl - Notebook 2: Task Classifier Training & Model Comparison
==================================================================

Trains 5 transformer models with identical settings for fair comparison.

Models: DistilBERT, BERT, ELECTRA, RoBERTa, DeBERTa
Expected: 97-98% accuracy with balanced data

Author: Kamala
Date: December 2024


In [2]:
#===============================================================================
# CELL 1: Setup
#===============================================================================

print("="*80)
print("MEMCTRL NOTEBOOK 2: TASK CLASSIFIER TRAINING")
print("="*80)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print("\nInstalling packages...")
!pip install -q --upgrade pip
!pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn

print("✓ Packages installed\n")

import sys
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score
import json
from pathlib import Path
import time
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Disable WandB
import os
os.environ['WANDB_DISABLED'] = 'true'

# GPU check
print("="*80)
print("GPU CHECK")
print("="*80)

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✓ READY\n" + "="*80)
else:
    print(" NO GPU! Enable T4 GPU in Runtime settings")
    sys.exit(1)

MEMCTRL NOTEBOOK 2: TASK CLASSIFIER TRAINING
Mounted at /content/drive

Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.6 MB/s eta 0:00:00
✓ Packages installed

GPU CHECK
✓ GPU: Tesla T4
✓ Memory: 15.8 GB

✓ READY


In [3]:
#===============================================================================
# CELL 2: Configuration
#===============================================================================

print("\n" + "="*80)
print("CONFIGURATION")
print("="*80)

CONFIG = {
    'data_path': '/content/drive/MyDrive/memctrl_data/labeled_dataset_cleaned.parquet',

    'output_dir': '/content/drive/MyDrive/memctrl_models/task_classifier_final',

    # Models
    'models': [
        {'name': 'distilbert-base-uncased', 'params': 66_000_000, 'batch': 16, 'grad_accum': 2},
        {'name': 'bert-base-uncased', 'params': 110_000_000, 'batch': 12, 'grad_accum': 3},
        {'name': 'google/electra-base-discriminator', 'params': 110_000_000, 'batch': 12, 'grad_accum': 3},
        {'name': 'roberta-base', 'params': 125_000_000, 'batch': 12, 'grad_accum': 3},
        {'name': 'microsoft/deberta-v3-base', 'params': 184_000_000, 'batch': 10, 'grad_accum': 3},
    ],

    # Training (IDENTICAL for all models)
    'max_length': 128,
    'num_epochs': 20,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.15,
    'lr_scheduler_type': 'cosine',
    'label_smoothing_factor': 0.1,
    'max_grad_norm': 1.0,

    # Evaluation
    'eval_steps': 300,
    'save_steps': 300,
    'logging_steps': 50,

    # Splits
    'train_split': 0.80,
    'val_split': 0.10,
    'test_split': 0.10,
    'seed': 42,
}

print("\n MODELS:")
for i, m in enumerate(CONFIG['models'], 1):
    print(f"  {i}. {m['name']} ({m['params']/1e6:.0f}M)")

print("\n SETTINGS:")
print(f"  LR: {CONFIG['learning_rate']}")
print(f"  Epochs: {CONFIG['num_epochs']} (NO early stopping)")
print(f"  Schedule: {CONFIG['lr_scheduler_type']}")

output_base = Path(CONFIG['output_dir'])
output_base.mkdir(parents=True, exist_ok=True)

with open(output_base / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)



CONFIGURATION

 MODELS:
  1. distilbert-base-uncased (66M)
  2. bert-base-uncased (110M)
  3. google/electra-base-discriminator (110M)
  4. roberta-base (125M)
  5. microsoft/deberta-v3-base (184M)

 SETTINGS:
  LR: 2e-05
  Epochs: 20 (NO early stopping)
  Schedule: cosine


In [4]:
#===============================================================================
# CELL 3: Load Dataset
#===============================================================================

print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

data_path = Path(CONFIG['data_path'])
if not data_path.exists():
    print(f" Dataset not found: {data_path}")
    print("\n Run Notebook 1 first!")
    sys.exit(1)

df = pd.read_parquet(data_path)
print(f"\n✓ Loaded {len(df):,} samples")
print(f"  Classes: {df['label'].nunique()}")

if 'category' in df.columns:
    print(f"  Categories: {df['category'].nunique()}")

# Validate
if 'prompt' not in df.columns or 'label' not in df.columns:
    print("Missing required columns")
    sys.exit(1)

if df[['prompt', 'label']].isnull().any().any():
    print("Found null values")
    sys.exit(1)

print("\n✓ Validation passed")



LOADING DATASET

✓ Loaded 17,190 samples
  Classes: 35
  Categories: 7

✓ Validation passed


In [5]:
#===============================================================================
# CELL 4: Label Mappings
#===============================================================================

print("\n" + "="*80)
print("LABEL MAPPINGS")
print("="*80)

unique_labels = sorted(df['label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

mappings = {
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},
    'num_labels': len(label2id),
}

with open(output_base / 'label_mappings.json', 'w') as f:
    json.dump(mappings, f, indent=2)

print(f"\n✓ {len(label2id)} classes")


LABEL MAPPINGS

✓ 35 classes


In [6]:
#===============================================================================
# CELL 5: Split Dataset
#===============================================================================

print("\n" + "="*80)
print("SPLITTING (80-10-10)")
print("="*80)

np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

dataset = Dataset.from_pandas(df[['prompt', 'label']])
total = len(dataset)

temp_size = CONFIG['val_split'] + CONFIG['test_split']
train_temp = dataset.train_test_split(test_size=temp_size, seed=CONFIG['seed'])
val_test = train_temp['test'].train_test_split(test_size=0.5, seed=CONFIG['seed'])

splits = {
    'train': train_temp['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

print("\n SPLITS:")
for name, split in splits.items():
    print(f"  {name.capitalize()}: {len(split):,} ({len(split)/total*100:.2f}%)")


SPLITTING (80-10-10)

 SPLITS:
  Train: 13,752 (80.00%)
  Val: 1,719 (10.00%)
  Test: 1,719 (10.00%)


In [7]:
#===============================================================================
# CELL 6: Metrics
#===============================================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(-1)

    accuracy = accuracy_score(labels, preds)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    kappa = cohen_kappa_score(labels, preds)

    return {
        'accuracy': accuracy,
        'f1_macro': f1_m,
        'f1_weighted': f1_w,
        'precision_macro': precision_m,
        'precision_weighted': precision_w,
        'recall_macro': recall_m,
        'recall_weighted': recall_w,
        'kappa': kappa,
    }

print("✓ Metrics defined")

✓ Metrics defined


In [8]:
#===============================================================================
# CELL 7: Training Function
#===============================================================================

def train_single_model(model_info, splits, label2id, id2label, output_base, config):
    name = model_info['name']
    display = name.split('/')[-1]

    print("\n" + "="*80)
    print(f"TRAINING: {display} ({model_info['params']/1e6:.0f}M)")
    print("="*80)

    model_dir = output_base / name.replace('/', '_')
    model_dir.mkdir(parents=True, exist_ok=True)

    start = time.time()

    try:
        batch = model_info['batch']
        grad_accum = model_info['grad_accum']

        print(f"\nSettings:")
        print(f"  LR: {config['learning_rate']}, Batch: {batch}, Grad: {grad_accum}")
        print(f"  Effective: {batch*grad_accum}")
        print(f"  Epochs: {config['num_epochs']} (NO early stop)")

        # Load
        print(f"\n[1/6] Loading...")
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForSequenceClassification.from_pretrained(
            name, num_labels=len(label2id), id2label=id2label, label2id=label2id
        )

        params = sum(p.numel() for p in model.parameters())
        print(f"✓ {params/1e6:.1f}M params")

        device = torch.device('cuda')
        model.to(device)

        # Tokenize
        print(f"\n[2/6] Tokenizing...")

        def tokenize(examples):
            tok = tokenizer(examples['prompt'], padding='max_length',
                          truncation=True, max_length=config['max_length'])
            tok['labels'] = [label2id[l] for l in examples['label']]
            return tok

        tokenized = {
            k: v.map(tokenize, batched=True, remove_columns=['prompt', 'label'])
            for k, v in splits.items()
        }
        print(f"✓ Done")

        # Setup
        print(f"\n[3/6] Setting up...")

        args = TrainingArguments(
            output_dir=str(model_dir),
            num_train_epochs=config['num_epochs'],
            per_device_train_batch_size=batch,
            per_device_eval_batch_size=batch * 2,
            learning_rate=config['learning_rate'],
            weight_decay=config['weight_decay'],
            warmup_ratio=config['warmup_ratio'],
            lr_scheduler_type=config['lr_scheduler_type'],
            label_smoothing_factor=config['label_smoothing_factor'],
            max_grad_norm=config['max_grad_norm'],
            fp16=True,
            gradient_accumulation_steps=grad_accum,
            eval_strategy='steps',  # CORRECT parameter name
            eval_steps=config['eval_steps'],
            save_strategy='steps',
            save_steps=config['save_steps'],
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            logging_dir=str(model_dir / 'logs'),
            logging_steps=config['logging_steps'],
            logging_first_step=True,
            report_to='none',
            dataloader_num_workers=0,
            seed=config['seed'],
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tokenized['train'],
            eval_dataset=tokenized['val'],
            compute_metrics=compute_metrics,
        )

        print(f"✓ Ready")

        # Train
        print(f"\n[4/6] Training...")
        print(f"Started: {datetime.now().strftime('%H:%M:%S')}")

        train_start = time.time()
        result = trainer.train()
        train_time = (time.time() - train_start) / 3600

        print(f"✓ Done: {train_time:.2f}h")

        # Save
        print(f"\n[5/6] Saving...")
        trainer.save_model(str(model_dir))
        tokenizer.save_pretrained(str(model_dir))
        print(f"✓ Saved")

        # Test
        print(f"\n[6/6] Testing...")
        test_results = trainer.predict(tokenized['test'])
        test_metrics = {k.replace('test_', ''): v for k, v in test_results.metrics.items()
                       if isinstance(v, (int, float))}

        val_results = trainer.evaluate(tokenized['val'])

        print(f"\nTest: {test_metrics['accuracy']*100:.2f}% acc, {test_metrics['f1_macro']*100:.2f}% F1")
        print(f"Val:  {val_results['eval_accuracy']*100:.2f}% acc, {val_results['eval_f1_macro']*100:.2f}% F1")

        # Inference
        print(f"\nInference speed...")
        times = []
        model.eval()
        with torch.no_grad():
            for _ in range(10):
                inputs = tokenizer("test", return_tensors='pt', max_length=128,
                                  truncation=True, padding='max_length').to(device)
                _ = model(**inputs)

            for _ in range(1000):
                inputs = tokenizer("test", return_tensors='pt', max_length=128,
                                  truncation=True, padding='max_length').to(device)
                t0 = time.time()
                _ = model(**inputs)
                torch.cuda.synchronize()
                times.append((time.time() - t0) * 1000)

        avg_inf = np.mean(times)
        p95_inf = np.percentile(times, 95)
        print(f"✓ {avg_inf:.2f}ms avg, {p95_inf:.2f}ms p95")

        # Results
        total_time = (time.time() - start) / 3600

        results = {
            'model_name': name,
            'display_name': display,
            'params': params,
            'params_millions': params / 1e6,
            'test_accuracy': test_metrics['accuracy'],
            'test_f1_macro': test_metrics['f1_macro'],
            'test_f1_weighted': test_metrics['f1_weighted'],
            'test_kappa': test_metrics['kappa'],
            'val_accuracy': val_results['eval_accuracy'],
            'val_f1_macro': val_results['eval_f1_macro'],
            'train_hours': train_time,
            'total_hours': total_time,
            'avg_inference_ms': avg_inf,
            'p95_inference_ms': p95_inf,
            'status': 'success',
        }

        with open(model_dir / 'results.json', 'w') as f:
            json.dump(results, f, indent=2)

        print(f"\n{'='*80}")
        print(f"✓ {display} DONE ({total_time:.2f}h)")
        print(f"  Test: {test_metrics['accuracy']*100:.2f}%")
        print(f"{'='*80}")

        del model, trainer
        torch.cuda.empty_cache()

        return results

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {display}")
        print(f"{'='*80}")
        print(f"{e}")

        import traceback
        traceback.print_exc()

        torch.cuda.empty_cache()

        return {
            'model_name': name,
            'display_name': display,
            'error': str(e),
            'status': 'failed',
        }

print("✓ Function defined")

✓ Function defined


In [ ]:
#===============================================================================
# CELL 8: Train All
#===============================================================================

print("\n" + "="*80)
print("TRAINING ALL MODELS")
print("="*80)

print(f"\nModels: {len(CONFIG['models'])}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Est: 12-15 hours")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

all_results = []

for i, model_info in enumerate(CONFIG['models'], 1):
    print(f"\n\n{'#'*80}")
    print(f"# {i}/{len(CONFIG['models'])}: {model_info['name'].split('/')[-1]}")
    print(f"{'#'*80}")

    result = train_single_model(model_info, splits, label2id, id2label, output_base, CONFIG)
    all_results.append(result)

    pd.DataFrame(all_results).to_csv(output_base / 'progress.csv', index=False)

    if result['status'] == 'success':
        best = max([r['test_accuracy'] for r in all_results if r['status']=='success'])
        print(f"\n[PROGRESS] {i}/{len(CONFIG['models'])}")
        print(f"Best: {best*100:.2f}%")

print(f"\n{'='*80}")
print("DONE!")
print(f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


TRAINING ALL MODELS

Models: 5
Epochs: 20
Est: 12-15 hours
Started: 2025-12-31 00:29:51


################################################################################
# 1/5: distilbert-base-uncased
################################################################################

TRAINING: distilbert-base-uncased (66M)

Settings:
  LR: 2e-05, Batch: 16, Grad: 2
  Effective: 32
  Epochs: 20 (NO early stop)

[1/6] Loading...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/13752 [00:00<?, ? examples/s]

Map:   0%|          | 0/1719 [00:00<?, ? examples/s]

Map:   0%|          | 0/1719 [00:00<?, ? examples/s]

✓ Done

[3/6] Setting up...
✓ Ready

[4/6] Training...
Started: 00:30:11


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,Kappa
300,3.001200,2.901451,0.297848,0.028566,0.149996,0.019701,0.101960,0.054419,0.297848,0.147653
600,2.271900,2.129613,0.550902,0.196981,0.470249,0.236900,0.471753,0.207999,0.550902,0.494233
900,1.578000,1.509236,0.764398,0.444808,0.736709,0.529191,0.756331,0.436224,0.764398,0.741208
1200,1.260400,1.173765,0.854567,0.601209,0.843243,0.659826,0.853689,0.598028,0.854567,0.841288
1500,1.053300,1.063339,0.887725,0.710882,0.883980,0.731861,0.884101,0.700304,0.887725,0.877665
1800,0.948100,1.020600,0.902850,0.785362,0.899854,0.811382,0.900810,0.768588,0.902850,0.893957
2100,0.889000,1.007621,0.903432,0.755595,0.900559,0.769440,0.900475,0.745715,0.903432,0.894539
2400,0.814000,0.975801,0.915649,0.782688,0.913811,0.781871,0.913645,0.792219,0.915649,0.908110
2700,0.760600,0.971343,0.916812,0.787329,0.914005,0.799004,0.913067,0.787551,0.916812,0.909270


In [ ]:
#===============================================================================
# CELL 9: Results
#===============================================================================

print("\n" + "="*80)
print("RESULTS")
print("="*80)

results_df = pd.DataFrame(all_results)
successful = results_df[results_df['status'] == 'success'].copy()

if len(successful) > 0:
    successful = successful.sort_values('test_f1_macro', ascending=False)
    successful.to_csv(output_base / 'all_results.csv', index=False)

    summary = pd.DataFrame({
        'Model': successful['display_name'],
        'Params (M)': successful['params_millions'].round(1),
        'Test Acc (%)': (successful['test_accuracy'] * 100).round(2),
        'Test F1 (%)': (successful['test_f1_macro'] * 100).round(2),
        'Val Acc (%)': (successful['val_accuracy'] * 100).round(2),
        'Kappa': successful['test_kappa'].round(3),
        'Train (h)': successful['train_hours'].round(2),
        'Infer (ms)': successful['avg_inference_ms'].round(2),
    })

    summary.to_csv(output_base / 'summary.csv', index=False)

    print("\n" + summary.to_string(index=False))

    best = summary.iloc[0]
    print(f"\n{'='*80}")
    print("BEST MODEL")
    print("="*80)
    print(f"  {best['Model']}")
    print(f"  Test: {best['Test Acc (%)']}%")
    print(f"  F1: {best['Test F1 (%)']}%")
    print("="*80)

    print(f"\n✓ Saved: summary.csv, all_results.csv")

In [ ]:
#===============================================================================
# CELL 10: Visualization
#===============================================================================

if len(successful) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Accuracy
    ax = axes[0, 0]
    x = np.arange(len(summary))
    w = 0.35
    ax.bar(x - w/2, summary['Test Acc (%)'], w, label='Acc', color='steelblue')
    ax.bar(x + w/2, summary['Test F1 (%)'], w, label='F1', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(summary['Model'], rotation=45, ha='right')
    ax.legend()
    ax.set_ylabel('Score (%)')
    ax.set_title('Performance')
    ax.grid(axis='y', alpha=0.3)

    # Size vs Acc
    ax = axes[0, 1]
    ax.scatter(summary['Params (M)'], summary['Test Acc (%)'], s=300, alpha=0.6)
    for _, row in summary.iterrows():
        ax.annotate(row['Model'], (row['Params (M)'], row['Test Acc (%)']),
                   fontsize=9, ha='center', va='bottom')
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Size vs Accuracy')
    ax.grid(alpha=0.3)

    # Training time
    ax = axes[1, 0]
    ax.barh(summary['Model'], summary['Train (h)'], color='purple', alpha=0.7)
    ax.set_xlabel('Hours')
    ax.set_title('Training Time')
    ax.grid(axis='x', alpha=0.3)

    # Inference
    ax = axes[1, 1]
    ax.barh(summary['Model'], summary['Infer (ms)'], color='green', alpha=0.7)
    ax.set_xlabel('Milliseconds')
    ax.set_title('Inference Speed')
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_base / 'comparison.png', dpi=300)
    print(f"\n✓ Saved: comparison.png")
    plt.show()

print("\n" + "="*80)
print("✓ COMPLETE")
print("="*80)

In [ ]:
# Run this to find labeling issues
from collections import Counter
import pandas as pd

# Load your data
df = pd.read_parquet('/content/drive/MyDrive/memctrl_data/labeled_dataset_final.parquet')

# Check for potential labeling errors
print("="*80)
print("LABEL QUALITY CHECKS")
print("="*80)

# 1. Find very similar prompts with different labels
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=1000)
vectors = vectorizer.fit_transform(df['prompt'])

# Sample check - find similar prompts with different labels
sample_idx = 0
similarities = cosine_similarity(vectors[sample_idx:sample_idx+1], vectors)[0]
similar_indices = similarities.argsort()[-20:][::-1]

print("\nSample similarity check:")
for idx in similar_indices[:10]:
    if idx != sample_idx:
        print(f"\nPrompt: {df.iloc[idx]['prompt'][:80]}")
        print(f"Label: {df.iloc[idx]['label']}")
        print(f"Similarity: {similarities[idx]:.2f}")

# 2. Check class distribution
print("\n" + "="*80)
print("CLASS DISTRIBUTION:")
print("="*80)
class_counts = df['label'].value_counts()
print(f"\nLargest: {class_counts.iloc[0]} samples")
print(f"Smallest: {class_counts.iloc[-1]} samples")
print(f"Ratio: {class_counts.iloc[0]/class_counts.iloc[-1]:.1f}:1")

# 3. Check for short/long outliers
print("\n" + "="*80)
print("PROMPT LENGTH ANALYSIS:")
print("="*80)
df['length'] = df['prompt'].str.len()
print(f"Mean length: {df['length'].mean():.0f} chars")
print(f"Median: {df['length'].median():.0f} chars")
print(f"Min: {df['length'].min()} chars")
print(f"Max: {df['length'].max()} chars")

print("\nShortest prompts:")
print(df.nsmallest(5, 'length')[['prompt', 'label', 'length']])

print("\nLongest prompts:")
print(df.nlargest(5, 'length')[['prompt', 'label', 'length']])

In [ ]:
# Load best model and analyze errors
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
import torch
import numpy as np

model_path = '/content/drive/MyDrive/memctrl_models/task_classifier_final/bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Get predictions on full dataset
from datasets import Dataset

test_dataset = Dataset.from_pandas(df[['prompt', 'label']])

def tokenize(examples):
    tok = tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)
    tok['labels'] = [label2id[l] for l in examples['label']]
    return tok

test_tokenized = test_dataset.map(tokenize, batched=True, remove_columns=['prompt', 'label'])

# Get predictions
trainer = Trainer(model=model)
predictions = trainer.predict(test_tokenized)

preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids
probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1)
confidences = probs.max(dim=-1).values

# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(labels, preds)

# Find most confused pairs
confusion_pairs = []
for i in range(len(label2id)):
    for j in range(len(label2id)):
        if i != j and cm[i][j] > 5:  # At least 5 confusions
            confusion_pairs.append({
                'true': id2label[i],
                'pred': id2label[j],
                'count': cm[i][j]
            })

confusion_pairs = sorted(confusion_pairs, key=lambda x: x['count'], reverse=True)

print("\n" + "="*80)
print("TOP 20 CONFUSION PAIRS:")
print("="*80)
for i, pair in enumerate(confusion_pairs[:20], 1):
    print(f"{i:2d}. {pair['true']:35s} → {pair['pred']:35s} ({pair['count']} errors)")

# High confidence errors
errors = []
for i in range(len(preds)):
    if preds[i] != labels[i]:
        errors.append({
            'index': i,
            'prompt': df.iloc[i]['prompt'],
            'true': id2label[labels[i]],
            'pred': id2label[preds[i]],
            'confidence': confidences[i].item(),
            'true_prob': probs[i][labels[i]].item(),
            'pred_prob': probs[i][preds[i]].item(),
        })

errors = sorted(errors, key=lambda x: x['confidence'], reverse=True)

print("\n" + "="*80)
print("TOP 20 HIGH-CONFIDENCE ERRORS:")
print("="*80)
for i, err in enumerate(errors[:20], 1):
    print(f"\n{i}. Confidence: {err['confidence']:.1%}")
    print(f"   Prompt: {err['prompt'][:120]}")
    print(f"   TRUE: {err['true']} (prob: {err['true_prob']:.1%})")
    print(f"   PRED: {err['pred']} (prob: {err['pred_prob']:.1%})")